# Module 2 Building a RAG Pipeline
### Ingest → Parse → Chunk → Embed → Index → Retrieve

**Prerequisites:** Module 1 notebook run to completion — you need an existing
`catalog.schema.volume` with at least one PDF in it.

⚠️ `ai_parse_document` and `ai_prep_search` are Beta SQL functions at the time of writing. Each
step below includes a **Fallback** cell that does the same job in plain PySpark if either isn't
enabled in your workspace yet.

## 1. Configuration — reuse Module 1's widgets

In [0]:
dbutils.widgets.text("catalog", "genai_course", "Catalog name")
dbutils.widgets.text("schema", "rag_demo", "Schema name")
dbutils.widgets.text("volume", "source_docs", "Volume name (raw files)")
dbutils.widgets.dropdown(
    "embedding_endpoint",
    "databricks-gte-large-en",
    ["databricks-gte-large-en", "databricks-bge-large-en"],
    "Embedding model endpoint",
)
dbutils.widgets.text("vs_endpoint", "genai_course_vs_endpoint", "Vector Search endpoint name")
dbutils.widgets.dropdown("chunk_size", "800", ["400", "800", "1200", "1600"], "Chunk size (characters)")
dbutils.widgets.dropdown("chunk_overlap_pct", "15", ["0", "10", "15", "25"], "Chunk overlap (%)")

print("✓ Widgets created — adjust chunk_size / chunk_overlap_pct any time and re-run Step 3 to re-chunk.")

In [0]:
CATALOG = "genai_course"
SCHEMA = "rag_demo"
VOLUME = "source_docs"

EMBEDDING_ENDPOINT = "databricks-gte-large-en"
VS_ENDPOINT = "genaicoursevsendpoint"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_TABLE = f"{CATALOG}.{SCHEMA}.rawdocuments"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.document_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.documentchunksindex"

print(f"PDF folder:       {VOLUME_PATH}")
print(f"Raw table:        {RAW_TABLE}")
print(f"Chunks table:     {CHUNKS_TABLE}")
print(f"Index name:       {INDEX_NAME}")
print(f"VS endpoint:      {VS_ENDPOINT}")
print(f"Embedding model:  {EMBEDDING_ENDPOINT}")

## ✅ Checkpoint 0 — Module 1 objects exist

In [0]:
try:
    files = dbutils.fs.ls(VOLUME_PATH)
    assert len(files) > 0
    print(f"✅ PASS — {len(files)} file(s) found in {VOLUME_PATH}")
    for f in files:
        print(f"   • {f.name}")
except Exception:
    print(f"❌ FAIL — {VOLUME_PATH} is missing or empty. Go finish Module 1 first.")

## 2. Parse the PDF into structure, not just text
`ai_parse_document` is layout-aware — it returns headings, tables, and reading order, not a flat
text blob.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {RAW_TABLE} AS
    SELECT
        path,
        ai_parse_document(content) AS parsed
    FROM READ_FILES('{VOLUME_PATH}', format => 'binaryFile')
    WHERE path ILIKE '%.pdf'
""")

print(f"✓ Parsed into {RAW_TABLE}")
display(spark.table(RAW_TABLE))

**Fallback** — if `ai_parse_document` isn't available in your workspace, run this instead (skip
the cell above):

In [0]:
# %pip install pypdf --quiet
# dbutils.library.restartPython()
#
# import pypdf
# from pyspark.sql.functions import udf
# from pyspark.sql.types import StringType
#
# def extract_text(path):
#     local_path = path.replace("dbfs:", "/dbfs")
#     reader = pypdf.PdfReader(local_path)
#     return "\n".join(page.extract_text() or "" for page in reader.pages)
#
# extract_text_udf = udf(extract_text, StringType())
# files_df = spark.sql(f"LIST '{VOLUME_PATH}'").filter("path ILIKE '%.pdf'")
# files_df.withColumn("text", extract_text_udf("path")).write.mode("overwrite").saveAsTable(RAW_TABLE)
# print(f"✓ Parsed (fallback) into {RAW_TABLE}")

## Look at what parsing actually produced
1. Click **Catalog** → your catalog → your schema → **Tables** → `raw_documents`.
2. Click **Sample Data** — this is the same preview `display()` showed you above, but from the UI.
3. Notice this table has **one row per PDF**, with the whole document's structure in a single
   `parsed` column — chunking happens next.

## 3. Chunk it
🧪 **This step reads the `chunk_size` / `chunk_overlap_pct` widgets from Step 1.** Change them and
re-run this cell to see how chunk count changes — no need to re-parse.

####Note that 
`ai_prep_search` is Beta, and access is gated on the workspace Previews page. Three clicks, but you need to be a workspace admin:

1. Click your username in the top bar of the workspace.
2. Select Previews from the menu.
3. Find **AI Prep Search** and flip the toggle on.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {CHUNKS_TABLE}
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
    AS
    SELECT
        path AS source_path,
        chunk:chunk_id::STRING AS chunk_id,
        chunk:chunk_to_embed::STRING AS chunk_to_embed,
        chunk:chunk_to_retrieve::STRING AS chunk_to_display
    FROM {RAW_TABLE}
    LATERAL VIEW EXPLODE(
        CAST(ai_prep_search(parsed):document:contents AS ARRAY<VARIANT>)
    ) AS chunk
""")

n_chunks = spark.table(CHUNKS_TABLE).count()
print(f"✓ {n_chunks} chunks written to {CHUNKS_TABLE}")
display(spark.table(CHUNKS_TABLE).limit(10))

**Fallback** — a plain Python UDF chunker, if `ai_prep_search` isn't available:

In [0]:
# from pyspark.sql.functions import udf, explode, monotonically_increasing_id, col
# from pyspark.sql.types import ArrayType, StringType
#
# def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
#     if not text:
#         return []
#     chunks, start = [], 0
#     while start < len(text):
#         chunks.append(text[start:start + size])
#         start += max(size - overlap, 1)
#     return chunks
#
# chunk_udf = udf(chunk_text, ArrayType(StringType()))
# chunks_df = (
#     spark.table(RAW_TABLE)
#     .withColumn("chunk_to_embed", explode(chunk_udf(col("text"))))
#     .withColumn("chunk_to_display", col("chunk_to_embed"))
#     .withColumn("chunk_id", monotonically_increasing_id())
# )
# chunks_df.write.mode("overwrite").option("delta.enableChangeDataFeed", "true").saveAsTable(CHUNKS_TABLE)
# print(f"✓ {chunks_df.count()} chunks written (fallback)")

## ✅ Checkpoint 1 — chunk table is real and CDF is on

In [0]:
props = {row.key: row.value for row in spark.sql(f"SHOW TBLPROPERTIES {CHUNKS_TABLE}").collect()}
cdf_on = props.get("delta.enableChangeDataFeed") == "true"
n_chunks = spark.table(CHUNKS_TABLE).count()

print("✅ PASS" if (cdf_on and n_chunks > 0) else "❌ FAIL", f"— {n_chunks} chunks, Change Data Feed={cdf_on}")
if not cdf_on:
    print("   Vector Search needs CDF to sync. Re-run the CREATE TABLE cell above.")

## 4. Create the AI Search endpoint and index
🖱️ **You can do this next part two ways — pick one:**
- **Via code** (cells below) — faster if you're doing this repeatedly.
- **Via UI** — click **Compute → AI Search → Create endpoint**, then from the
  `document_chunks` table's page click **Create index**, and fill in the same fields the code
  below sets programmatically. Either path produces the same object.

We'll do it via code here so the notebook is fully runnable, but try the UI path once so you know
where to find it later.

In [0]:
%pip install --upgrade --force-reinstall databricks-ai-search
dbutils.library.restartPython()

Now rerun cell 4 again after the restart

In [0]:
from databricks.ai_search.client import AISearchClient

vsc = AISearchClient()

response = vsc.list_endpoints()
existing_endpoints = [
    endpoint["name"]
    for endpoint in response.get("endpoints", [])
]

if VS_ENDPOINT not in existing_endpoints:
    vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD"
    )
    print(
        f"⏳ Creating endpoint {VS_ENDPOINT} — "
        "this can take several minutes."
    )
else:
    print(f"✓ Reusing existing endpoint: {VS_ENDPOINT}")

In [0]:
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=CHUNKS_TABLE,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_to_embed",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
        columns_to_sync=[
            "source_path",
            "chunk_to_display",
        ],
    )

    print(f"⏳ Index requested: {INDEX_NAME}")

except Exception as e:
    if "already exists" in str(e).lower():
        index = vsc.get_index(
            endpoint_name=VS_ENDPOINT,
            index_name=INDEX_NAME,
        )

        print(f"✓ Index already exists: {INDEX_NAME}")
        index.sync()
        print("⏳ Sync requested.")

    else:
        raise

print(index.describe())

## Watch it sync, live
1. Click **Compute** → **AI Search** in the sidebar.
2. Click your endpoint, then your index.
3. Watch the **status** field — it moves from provisioning → syncing → **Ready**. First sync is
   usually the slowest step in this whole module; a coffee break is well timed here.

## ✅ Checkpoint 2 — poll until ready, right here in the notebook
This does the same thing as refreshing the UI page, just without leaving the notebook.

In [0]:
import time

index_handle = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)

max_attempts = 30
for attempt in range(max_attempts):
    status = index_handle.describe().get("status", {})
    ready = status.get("ready", False)
    bar = "█" * (attempt + 1) + "░" * (max_attempts - attempt - 1)
    print(f"\r[{bar}] ready={ready} {status.get('message', '')}", end="", flush=True)
    if ready:
        print("\n✅ PASS — index is ready")
        break
    time.sleep(20)
else:
    print("\n⚠️  Still not ready after 10 minutes — check the UI for an error message.")

## 5. Search it — change the question and re-run
Run the widget cell once, then edit `search_query` above the next cell and re-run just that cell,
as many times as you like.

In [0]:
dbutils.widgets.text("search_query", "What are the Variants of Gradient Descent?", "Try a search query")
dbutils.widgets.dropdown("search_type", "ANN", ["ANN", "HYBRID"], "Search type")

In [0]:
query = dbutils.widgets.get("search_query")
search_type = dbutils.widgets.get("search_type")

results = index_handle.similarity_search(
    query_text=query,
    columns=[
        "chunk_to_display",
        "source_path",
    ],
    num_results=5,
    query_type=search_type,
)

rows = results.get("result", {}).get("data_array", [])

print(f"Query: {query}")
print(f"Search type: {search_type}")
print(f"Retrieved chunks: {len(rows)}")

for i, row in enumerate(rows, start=1):
    print(f"\n--- Result {i} ---")
    print(f"Score: {row[-1]:.3f}")
    print(f"Source: {row[1]}")
    print(row[0][:800])

👆 **Try both `search_type` values on the same query.** `HYBRID` blends keyword and vector
matching — notice whether the ranking changes for questions with exact product codes or names in
them versus purely conceptual questions.

## 6. Wire retrieval + generation into a traced RAG chain
No framework — plain Python. Because tracing is already on, every call inside these functions
becomes its own span automatically.

In [0]:
import mlflow
from openai import OpenAI

mlflow.openai.autolog()


# Connect to the Databricks AI Gateway / Model Serving endpoint.
def get_chat_client():
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    workspace_url = ctx.apiUrl().get()
    token = ctx.apiToken().get()

    return OpenAI(
        api_key=token,
        base_url=f"{workspace_url}/ai-gateway/mlflow/v1",
    )


openai_client = get_chat_client()

# Keep this only if this model endpoint exists in your workspace.
LLM_ENDPOINT = "system.ai.gpt-oss-20b"


# Load the index that you created earlier.
index_handle = vsc.get_index(
    endpoint_name=VS_ENDPOINT,
    index_name=INDEX_NAME,
)


@mlflow.trace(span_type="RETRIEVER")
def retrieve(question: str, k: int = 5):
    results = index_handle.similarity_search(
        query_text=question,
        columns=[
            "chunk_to_display",
            "source_path",
        ],
        num_results=k,
        query_type="HYBRID",
    )

    rows = results.get("result", {}).get("data_array", [])

    # The first returned column is chunk_to_display.
    return [row[0] for row in rows]


@mlflow.trace(span_type="CHAIN")
def rag_answer(question: str):
    context_chunks = retrieve(question)
    context = "\n\n---\n\n".join(context_chunks)

    prompt = f"""You are a helpful research assistant.

Answer the question using ONLY the retrieved context below.
If the answer is not present in the context, say:
"I could not find that information in the uploaded PDF."

Retrieved context:
{context}

Question: {question}
"""

    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        max_tokens=400,
        temperature=0.2,
    )

    return response.choices[0].message.content


print("✓ rag_answer() is ready")

## Try it yourself — full RAG, end to end

In [0]:
dbutils.widgets.text("rag_question", "What are the loss functions?", "Ask your RAG chain")

In [0]:
question = dbutils.widgets.get("rag_question")
answer = rag_answer(question)
print(f"Q: {question}\n\nA: {answer}")

## Read the full trace
1. **Experiments → Traces**, open the most recent run.
2. You should see a **CHAIN** span containing a **RETRIEVER** span and a chat-completion span,
   nested — click into the RETRIEVER span specifically and confirm the chunks it returned are
   actually relevant to your question.
3. If the answer looks wrong, this is where you'd look first: bad retrieval shows up here before
   it shows up as a bad answer.

---
### What's next
**Module 3** takes `INDEX_NAME` from this notebook and wraps it as just one tool among several —
adding table lookups and multistep reasoning on top of the retrieval you just built.